# Azure SQL Integration - Master Data Load

Loads card, customer and merchant master CSVs from the UC volume into Azure SQL Database (`capstone1team1creditcard`) over JDBC, then reads them back to verify.

In [0]:
# Azure SQL connection details (credentials come from the secret scope, never hard-coded)
server = "capstone1dbserver.database.windows.net"
database = "capstone1team1creditcard"

jdbc_url = (
    f"jdbc:sqlserver://{server}:1433;"
    f"databaseName={database};"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

properties = {
    "user": dbutils.secrets.get("azure-sql-scope", "sql-user"),
    "password": dbutils.secrets.get("azure-sql-scope", "sql-pwd"),
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

base_path = "/Volumes/capstone_team01/bronze/raw_data/datasets/master_data/azure_sql_seed"
tables = ["card_master", "customer_master", "merchant_master"]

In [0]:
dfs = {}
for t in tables:
    dfs[t] = (spark.read
              .option("header", True)
              .option("inferSchema", True)
              .csv(f"{base_path}/{t}.csv"))
    print(f"{t}: {dfs[t].count()} rows")

display(dfs["card_master"].limit(5))

In [0]:
# Serverless only allows writes via the "sqlserver" format (generic .jdbc() writes are blocked).
# overwrite + truncate keeps the existing table definitions created in Azure SQL.
for t in tables:
    (dfs[t].write
        .format("sqlserver")
        .mode("overwrite")
        .option("truncate", "true")
        .option("host", server)
        .option("port", "1433")
        .option("database", database)
        .option("dbtable", f"dbo.{t}")
        .option("user", properties["user"])
        .option("password", properties["password"])
        .save())
    print(f"Loaded dbo.{t}")

In [0]:
rows = []
for t in tables:
    sql_count = spark.read.jdbc(jdbc_url, f"(SELECT COUNT(*) AS c FROM dbo.{t}) x", properties=properties).first()[0]
    rows.append((t, dfs[t].count(), sql_count))

display(spark.createDataFrame(rows, "table_name string, csv_rows long, azure_sql_rows long"))
display(spark.read.jdbc(jdbc_url, "dbo.customer_master", properties=properties).limit(5))